## Instalación de dependencias (en dos pasos para evitar conflictos con `requests`)

In [ ]:
# 1. Instalar dependencias
!pip -q install -U fastapi uvicorn pyngrok langchain langchain-openrouter

In [ ]:
!pip -q install requests==2.32.4 --upgrade --force-reinstall --no-deps

**Nota:** Si al instalar requests te da un error, vuelve a ejecutar la celda.

## Imports y configuración de secretos

In [ ]:
# 2. Importar librerías y configurar
import os
import sqlite3
import re
import json
import tempfile
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import List, Dict, Any
from google.colab import userdata
from langchain_openrouter import ChatOpenRouter

In [ ]:
# Configurar OpenRouter
openrouter_api_key = userdata.get("OPENROUTER_API_KEY")
if not openrouter_api_key:
    raise ValueError("Agrega OPENROUTER_API_KEY en los Secrets de Colab.")
os.environ["OPENROUTER_API_KEY"] = openrouter_api_key

In [ ]:
# Crear el modelo de generación
llm = ChatOpenRouter(
    model="google/gemini-2.5-flash-lite",
    temperature=0.1,  # Baja temperatura para respuestas más deterministas (SQL exacto)
)

In [ ]:
# 3. Crear la aplicación FastAPI
app = FastAPI(
    title="Generador de consultas SQL con lenguaje natural",
    description="API que traduce preguntas en español a SQL y ejecuta consultas de solo lectura con validación.",
    version="1.0.0",
)

## Crear la base de datos y poblarla con datos de ejemplo

In [ ]:
# 4. Crear base de datos SQLite en un archivo temporal

# Crear un archivo temporal para la base de datos
db_path = os.path.join(tempfile.gettempdir(), 'cursos.db')

# Eliminar la base de datos anterior si existe (para empezar limpio)
if os.path.exists(db_path):
    os.remove(db_path)

# Crear conexión inicial y las tablas
conn = sqlite3.connect(db_path)

# Activar el cumplimiento de claves foráneas
conn.execute("PRAGMA foreign_keys = ON")

cursor = conn.cursor()

In [ ]:
# Crear tablas (con relaciones)
cursor.execute("""
CREATE TABLE instructores (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT NOT NULL,
    especialidad TEXT,
    email TEXT
)
""")

cursor.execute("""
CREATE TABLE cursos (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    titulo TEXT NOT NULL,
    descripcion TEXT,
    categoria TEXT,
    nivel TEXT,
    duracion_horas INTEGER,
    precio REAL,
    fecha_creacion DATE,
    instructor_id INTEGER,
    FOREIGN KEY (instructor_id) REFERENCES instructores(id)
)
""")

cursor.execute("""
CREATE TABLE inscripciones (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    curso_id INTEGER,
    estudiante TEXT NOT NULL,
    fecha_inscripcion DATE,
    FOREIGN KEY (curso_id) REFERENCES cursos(id)
)
""")

In [ ]:
# Insertar instructores primero (porque cursos referencia a instructores)
instructores_data = [
    ('Ana Martínez', 'Programación', 'ana@example.com'),
    ('Carlos García', 'Datos', 'carlos@example.com'),
    ('Laura Pérez', 'IA', 'laura@example.com'),
    ('Juan Gómez', 'DevOps', 'juan@example.com'),
    ('María López', 'Marketing', 'maria@example.com'),
]

cursor.executemany('''
INSERT INTO instructores (nombre, especialidad, email)
VALUES (?, ?, ?)
''', instructores_data)

In [ ]:
# Insertar cursos (con instructor_id)
cursos_data = [
    ('Python básico', 'Introducción a Python para principiantes', 'Programación', 'Principiante', 20, 49.99, '2024-01-15', 1),  # Ana Martínez
    ('JavaScript avanzado', 'Profundiza en JavaScript y frameworks modernos', 'Programación', 'Intermedio', 30, 79.99, '2024-02-01', 1),  # Ana Martínez
    ('SQL para análisis de datos', 'Aprende SQL desde cero con ejemplos prácticos', 'Datos', 'Principiante', 15, 39.99, '2024-02-10', 2),  # Carlos García
    ('Machine Learning aplicado', 'Introducción a ML con Python y scikit-learn', 'Datos', 'Intermedio', 40, 99.99, '2024-03-01', 3),  # Laura Pérez
    ('Desarrollo web con Flask', 'Crea aplicaciones web con Flask y Python', 'Programación', 'Intermedio', 25, 59.99, '2024-03-15', 1),  # Ana Martínez
    ('Inteligencia Artificial con Python', 'Fundamentos de IA con proyectos prácticos', 'IA', 'Avanzado', 50, 129.99, '2024-04-01', 3),  # Laura Pérez
    ('Diseño de bases de datos', 'Modelado y diseño de bases de datos relacionales', 'Datos', 'Intermedio', 20, 69.99, '2024-04-10', 2),  # Carlos García
    ('DevOps con Docker y Kubernetes', 'Introducción a la contenerización y orquestación', 'DevOps', 'Avanzado', 35, 89.99, '2024-05-01', 4),  # Juan Gómez
    ('Marketing digital con IA', 'Aplica IA para estrategias de marketing', 'Marketing', 'Principiante', 12, 34.99, '2024-05-15', 5),  # María López
    ('Ciberseguridad básica', 'Principios de seguridad informática', 'Seguridad', 'Principiante', 18, 44.99, '2024-06-01', 4),  # Juan Gómez
]

cursor.executemany('''
INSERT INTO cursos (titulo, descripcion, categoria, nivel, duracion_horas, precio, fecha_creacion, instructor_id)
VALUES (?, ?, ?, ?, ?, ?, ?, ?)
''', cursos_data)

In [ ]:
inscripciones_data = [
    (1, 'Estudiante A', '2024-01-20'),
    (1, 'Estudiante B', '2024-01-22'),
    (2, 'Estudiante C', '2024-02-05'),
    (2, 'Estudiante D', '2024-02-07'),
    (3, 'Estudiante E', '2024-02-15'),
    (3, 'Estudiante F', '2024-02-18'),
    (4, 'Estudiante G', '2024-03-10'),
    (4, 'Estudiante H', '2024-03-12'),
    (5, 'Estudiante I', '2024-03-20'),
    (6, 'Estudiante J', '2024-04-05'),
    (7, 'Estudiante K', '2024-04-15'),
    (8, 'Estudiante L', '2024-05-05'),
    (9, 'Estudiante M', '2024-05-20'),
    (10, 'Estudiante N', '2024-06-05'),
]

cursor.executemany('''
INSERT INTO inscripciones (curso_id, estudiante, fecha_inscripcion)
VALUES (?, ?, ?)
''', inscripciones_data)

In [ ]:
conn.commit()
conn.close()

print("✅ Base de datos creada con datos de ejemplo.")
print(f"📊 Cursos: {len(cursos_data)}, Instructores: {len(instructores_data)}, Inscripciones: {len(inscripciones_data)}")

## Función para obtener una conexión a la base de datos

In [ ]:
# Función para obtener una nueva conexión a la base de datos
def get_db_connection():
    """
    Crea una conexión de solo lectura a la base de datos SQLite.

    La base ya fue creada y poblada durante la preparación del notebook.
    Los endpoints únicamente necesitan consultar sus datos.
    """
    database_uri = f"file:{db_path}?mode=ro"

    conn = sqlite3.connect(
        database_uri,
        uri=True
    )

    conn.row_factory = sqlite3.Row
    return conn

## Definir los modelos Pydantic para este proyecto

In [ ]:
# 5. Modelos Pydantic
class QueryRequest(BaseModel):
    question: str = Field(..., min_length=3, description="Pregunta en español sobre la base de datos")

class QueryResponse(BaseModel):
    question: str
    sql_query: str
    results: List[Dict[str, Any]]
    row_count: int
    explanation: str

class ColumnInfo(BaseModel):
    name: str
    type: str

class TableInfo(BaseModel):
    name: str
    columns: List[ColumnInfo]

class SchemaResponse(BaseModel):
    tables: List[TableInfo]

## Implementar el endpoint `GET /schema`

In [ ]:
# 6. Endpoint GET /schema
@app.get("/schema", response_model=SchemaResponse)
def get_schema():
    """
    Devuelve la estructura de la base de datos (tablas, columnas y tipos).
    """
    conn = None
    try:
        # Crear una nueva conexión para esta petición
        conn = get_db_connection()
        cursor = conn.cursor()

        tables = []
        # Obtener lista de tablas
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'")
        table_names = [row[0] for row in cursor.fetchall()]

        for table_name in table_names:
            # Obtener información de columnas
            cursor.execute(f"PRAGMA table_info({table_name})")
            columns_info = cursor.fetchall()
            columns = [ColumnInfo(name=col[1], type=col[2]) for col in columns_info]
            tables.append(TableInfo(name=table_name, columns=columns))

        return SchemaResponse(tables=tables)
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Error al obtener el esquema: {str(e)}")
    finally:
        # Siempre cerrar la conexión después de usarla
        if conn:
            conn.close()

## Función para obtener el esquema como texto (para el prompt)

In [ ]:
# 7. Función auxiliar para obtener esquema en texto (mejorado con relaciones)
def get_schema_text() -> str:
    """
    Devuelve el esquema de la base de datos en formato texto
    para incluirlo en el prompt del modelo.
    """
    conn = None
    try:
        conn = get_db_connection()
        cursor = conn.cursor()

        cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'")
        table_names = [row[0] for row in cursor.fetchall()]

        schema_lines = []
        for table_name in table_names:
            cursor.execute(f"PRAGMA table_info({table_name})")
            columns = cursor.fetchall()
            col_defs = []
            for col in columns:
                col_defs.append(f"  {col[1]} {col[2]}")
            schema_lines.append(f"Tabla {table_name}:\n" + "\n".join(col_defs))

        # Añadir información sobre relaciones importantes
        relations = """
Relaciones importantes entre tablas:
- cursos.instructor_id → instructores.id (cada curso tiene un instructor)
- inscripciones.curso_id → cursos.id (cada inscripción pertenece a un curso)
"""
        schema_text = "\n\n".join(schema_lines) + "\n\n" + relations

        return schema_text
    finally:
        if conn:
            conn.close()

## Endpoint `/query` (generación, validación y ejecución)

In [ ]:
# 8. Endpoint POST /query
@app.post("/query", response_model=QueryResponse)
def query(request: QueryRequest):
    """
    Recibe una pregunta en español, genera SQL, valida y ejecuta la consulta.
    """
    conn = None
    try:
        # 1. Obtener el esquema de la base de datos
        schema_text = get_schema_text()

        # 2. Construir el prompt para el modelo
        prompt = f"""
Eres un asistente que genera consultas SQL basadas en preguntas en español.

Base de datos SQLite con las siguientes tablas:

{schema_text}

Instrucciones importantes:
- Genera SOLO la consulta SQL, sin texto adicional, sin markdown, sin comentarios.
- La consulta debe ser SOLO SELECT (no INSERT, UPDATE, DELETE, DROP, ALTER, etc.).
- Usa los nombres de columnas exactos como aparecen en el esquema.
- Si la pregunta no tiene sentido o no se puede responder con la base de datos, responde: \"No se puede generar una consulta válida para esta pregunta.\"

Pregunta del usuario: {request.question}

SQL:
"""

        # 3. Llamar al modelo
        response = llm.invoke(prompt)
        sql_query = response.content.strip()

        # 4. Limpiar la consulta (remover posibles bloques de código)
        if sql_query.startswith("```sql"):
            sql_query = sql_query[6:]
        if sql_query.startswith("```"):
            sql_query = sql_query[3:]
        if sql_query.endswith("```"):
            sql_query = sql_query[:-3]

        sql_query = sql_query.strip()
        sql_query = sql_query.rstrip(";").strip()

        # No permitir múltiples sentencias
        if ";" in sql_query:
            raise HTTPException(
                status_code=400,
                detail="Solo se permite una sentencia SQL por petición."
            )

        # Validación básica: solo SELECT
        if not sql_query.upper().startswith("SELECT"):
            raise HTTPException(
                status_code=400,
                detail=f"La consulta generada no es SELECT: {sql_query}"
            )

        forbidden_words = ["INSERT", "UPDATE", "DELETE", "DROP", "ALTER", "CREATE", "REPLACE", "TRUNCATE"]
        pattern = r"\\b(" + "|".join(forbidden_words) + r")\\b"

        if re.search(pattern, sql_query, re.IGNORECASE):
            raise HTTPException(
                status_code=400,
                detail=f"La consulta contiene una operación no permitida: {sql_query}"
            )

        # 6. Ejecutar la consulta
        try:
            conn = get_db_connection()
            cursor = conn.cursor()
            cursor.execute(sql_query)
            results = cursor.fetchall()

            # Obtener nombres de columnas
            column_names = [description[0] for description in cursor.description] if cursor.description else []

            # Convertir resultados a lista de diccionarios
            formatted_results = [dict(zip(column_names, row)) for row in results]

            # Generar explicación breve
            explanation = f"Se encontraron {len(formatted_results)} resultados para la consulta."
            if len(formatted_results) == 0:
                explanation = "No se encontraron resultados para la consulta."

            return QueryResponse(
                question=request.question,
                sql_query=sql_query,
                results=formatted_results,
                row_count=len(formatted_results),
                explanation=explanation
            )

        except sqlite3.Error as e:
            raise HTTPException(
                status_code=400,
                detail=f"Error al ejecutar la consulta SQL: {str(e)}"
            )

    except HTTPException:
        raise
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Error al procesar la pregunta: {str(e)}")
    finally:
        if conn:
            conn.close()

## Levantar la API y exponerla con ngrok

In [ ]:
# 9. Levantar servidor
import threading
import time
import uvicorn

PORT = 8000

def run_api():
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="info")

server_thread = threading.Thread(target=run_api, daemon=True)
server_thread.start()
time.sleep(2)
print(f"✅ Servidor iniciado en el puerto {PORT}")

In [ ]:
# 10. Abrir túnel con ngrok
from pyngrok import ngrok

ngrok_authtoken = userdata.get("NGROK_AUTHTOKEN")
if not ngrok_authtoken:
    raise ValueError("Agrega NGROK_AUTHTOKEN en los Secrets de Colab.")

ngrok.set_auth_token(ngrok_authtoken)
ngrok.kill()
tunnel = ngrok.connect(PORT, "http")
public_url = tunnel.public_url

print("🌍 URL pública temporal:")
print(public_url)
print("\n📚 Documentación interactiva (Swagger UI):")
print(public_url + "/docs")

## Prueba desde el notebook con `requests`

In [ ]:
import requests
import json

headers = {"ngrok-skip-browser-warning": "true"}
base_url = public_url

# 1. Ver el esquema
resp = requests.get(f"{base_url}/schema", headers=headers)
print("📊 Esquema:")
print(json.dumps(resp.json(), indent=2, ensure_ascii=False)[:500] + "...")

In [ ]:
# 2. Hacer una pregunta simple
payload = {"question": "¿Cuáles son los cursos de programación?"}
resp = requests.post(f"{base_url}/query", json=payload, headers=headers)
print("\n🔍 Pregunta:", payload["question"])
print("📝 SQL:", resp.json()["sql_query"])
print("📋 Resultados:")
for row in resp.json()["results"]:
    print(f"   - {row.get('titulo', row)}")
print(f"   ({resp.json()['row_count']} resultados)")

In [ ]:
# 3. Pregunta con JOIN (manejando errores correctamente)
payload = {"question": "¿Qué cursos imparte el instructor Ana Martínez?"}
resp = requests.post(f"{base_url}/query", json=payload, headers=headers)

print("\n🔍 Pregunta:", payload["question"])

if resp.status_code == 200:
    data = resp.json()
    print("📝 SQL:", data.get("sql_query", "No se generó SQL"))
    print("📋 Resultados:")
    for row in data.get("results", []):
        print(f"   - {row.get('titulo', row)}")
    print(f"   ({data.get('row_count', 0)} resultados)")
else:
    # Mostrar el error de forma clara
    print("❌ Error:", resp.json().get("detail", resp.text))